In [6]:
import pandas as pd
from gqlalchemy import Memgraph
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

feature_names = [
    "tx_amount", "tx_velocity_1h", "fan_out_degree", "fan_in_degree",
    "pagerank_score", "betweenness_centrality", "louvain_community_id",
    "avg_hop_distance", "account_age_days", "cross_border_flag",
    "device_risk_score", "ip_entropy"
]

def load_data():
    try:
        memgraph = Memgraph(host="127.0.0.1", port=7687)
        # Fetch properties explicitly from node/relationship maps
        query = """
        MATCH (a:Account)-[t:TRANSACTION]->(b:Account)
        RETURN 
            coalesce(t.amount, 0.0) AS tx_amount,
            coalesce(a.velocity_1h, 0.0) AS tx_velocity_1h,
            coalesce(a.fan_out, 0) AS fan_out_degree,
            coalesce(a.fan_in, 0) AS fan_in_degree,
            coalesce(a.pagerank, 0.0) AS pagerank_score,
            coalesce(a.betweenness, 0.0) AS betweenness_centrality,
            coalesce(a.community_id, 0) AS louvain_community_id,
            coalesce(a.avg_hop, 0.0) AS avg_hop_distance,
            coalesce(a.age_days, 0) AS account_age_days,
            coalesce(t.is_cross_border, 0) AS cross_border_flag,
            coalesce(a.device_risk, 0.0) AS device_risk_score,
            coalesce(a.ip_entropy, 0.0) AS ip_entropy,
            coalesce(t.is_fraud, 0) AS is_fraud
        """
        results = list(memgraph.execute_and_fetch(query))
        
        if len(results) > 0:
            df = pd.DataFrame(results)
            # Normalize dictionary results if returned as mgclient objects
            if isinstance(results[0], dict) and 'is_fraud' in df.columns:
                print("Successfully loaded transaction feature matrix from Memgraph.")
                return df
                
        print("Memgraph query returned empty or unparseable graph records. Generating synthetic data...")
    except Exception as e:
        print(f"Could not connect or query Memgraph ({e}). Generating synthetic benchmark data...")

    # Fallback to balanced/synthetic data generation
    X, y = make_classification(
        n_samples=5000, n_features=12, n_informative=8, n_redundant=2,
        weights=[0.97, 0.03], random_state=42
    )
    df = pd.DataFrame(X, columns=feature_names)
    df['is_fraud'] = y
    return df

df_features = load_data()

X = df_features[feature_names]
y = df_features['is_fraud'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Set: {len(X_train)} transactions | Test Set: {len(X_test)} transactions")
print(f"Fraud distribution: {y.sum()} fraudulent / {len(y) - y.sum()} legitimate")

Memgraph query returned empty or unparseable graph records. Generating synthetic data...
Training Set: 4000 transactions | Test Set: 1000 transactions
Fraud distribution: 172 fraudulent / 4828 legitimate


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)
from sklearn.datasets import make_classification
import xgboost as xgb
import shap

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Environment initialized successfully.")


Environment initialized successfully.


c:\Users\Hp\fraudnet-zero\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
